### Manual: 10-config

Run this cell in the Kaggle notebook UI. Environment persists across cells in the same session (scripts in `/kaggle/working/smoke_test`, venv, repo clone under `/root/bevfusion_smoke`).


In [ ]:
import os, sys
WORK = os.environ.get('SMOKE_WORK', '/kaggle/working/smoke_test')
if not os.path.exists(os.path.join(WORK, 'smoke_common.py')):
    raise SystemExit('Run 01-bootstrap first in this session (or run the full notebook once to create scripts).')
sys.path.insert(0, WORK)
print('guard ok:', WORK)


In [ ]:
"""Resolve the official config chain (no torchpack) and check its invariants.

Runs `smoke_cfg.py` in the pinned venv: merges `default.yaml` from every ancestor
directory of the leaf config (torchpack semantics), evaluates every `${...}`
expression, writes `artifacts/cfg_resolved.json` + `artifacts/config_snapshot.json`.
"""

import json
import os
import sys

WORK = os.environ.get("SMOKE_WORK", "/kaggle/working/smoke_test")
sys.path.insert(0, WORK)
import smoke_common as sc  # noqa: E402

rc, text = sc.run_script("smoke_cfg.py", log_name="smoke_cfg.log", timeout=600)
print()
print("rc = %d" % rc)
checks = sc.load_json("cfg_checks.json")
snapshot = sc.load_json("config_snapshot.json")
print("config chain:")
for entry in snapshot.get("chain", []):
    print("  %-90s %s" % (entry["path"], entry["sha256"][:16]))
print()
print("resolved values:")
for key in (
    "image_size",
    "point_cloud_range",
    "voxel_size",
    "vtransform_type",
    "vtransform_out_channels",
    "vtransform_dbound",
    "vtransform_feature_size",
    "fuser_in_channels",
    "lidar_sparse_shape",
    "head_test_grid_size",
):
    print("  %-28s %s" % (key, json.dumps(checks.get(key))))
print()
print("failed invariants: %s" % json.dumps(checks.get("failed")))
print("CONFIG_CELL_OK" if checks.get("ok") else "CONFIG_CELL_FAILED")
